## 6.6 池化层（Pooling Layer） - 在现代网络中的使用变化

##### 1. 为什么要讨论“使用变化”

##### 1.1 早期 CNN 中，池化层非常常见
在早期经典 CNN 结构中，池化层几乎是标准配置。

例如常见模式是：

`卷积 → 激活 → 池化`

再重复几次。

这是因为早期网络通常希望通过池化层来：
* 压缩特征图
* 降低计算量
* 增强一定的平移鲁棒性

##### 1.2 但现代网络设计思路更灵活了
随着 CNN 结构不断发展，大家发现：

下采样不一定只能靠传统池化层完成。

于是现代网络中，池化层的使用方式开始发生变化：
* 有些地方继续使用池化
* 有些地方减少池化
* 有些地方改用 stride 卷积
* 有些地方只在最后保留全局池化

所以这一节的重点不是说“池化层过时了”，

而是要理解：

现代网络对池化层的使用变得更有选择性了。

#### 2. 核心原因：不再每一阶段都机械使用池化层

##### 2.1 早期风格
早期网络常常是这样：

`Conv → ReLU → Pool → Conv → ReLU → Pool`

也就是说，每提取完一层或一组特征，就接一次池化。

##### 2.2 现代风格
现代网络中，设计往往更灵活。

有时不会每个阶段都固定接池化，

而是根据任务和结构需求决定：
* 哪一层需要下采样
* 哪一层保持分辨率
* 哪一层更适合用卷积替代池化

这说明池化层不再是“必须每层都接”的固定模块。

#### 3. 变化一：不使用池化层 下采样改用 stride 卷积完成

##### 3.1 什么是 stride 卷积下采样
我们前面学过，卷积层中的 stride > 1 也能让输出尺寸变小。

例如：

`nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)`

这既能：
* 提取特征
* 又能把空间尺寸缩小

所以它本身就同时完成了：

>特征提取 + 下采样

##### 3.2 为什么现代网络喜欢这样做
因为传统池化层只是压缩，

它通常没有可学习参数。

而 stride 卷积则不同，它在压缩的同时还能学习更合适的变换方式。

所以现代网络常常会觉得：

与其固定规则地下采样，不如让网络自己学着下采样。

##### 3.3 这带来的变化
于是很多现代 CNN 中会看到：
* 某些地方不用 Max Pooling
* 而改成 stride = 2 的卷积层

这是一种非常常见的变化趋势。

#### 4. 变化二：不采用 Flatten，使用GAP代替

##### 4.1 早期网络后部
早期 CNN 在最后常常这样做：
* 卷积层输出
* 池化层MaxPooling
* Flatten
* 多个全连接层
* 输出分类结果

这种方式的问题是：
* 参数量大
* 更容易过拟合
* 计算开销更高

##### 4.2 现代网络后部
现代 CNN 更常见的做法是：
* 卷积层输出
* 池化层Global Average Pooling
* 小型分类层

也就是说，很多现代网络会用：

>GAP 替代大规模 Flatten + FC 的结构

##### 4.3 为什么 GAP 更受欢迎
因为它有几个优势：
* 参数更少
* 结构更简洁
* 更不容易过拟合
* 更符合“每个通道总结一个高级特征”的思路

所以在现代网络中：

局部池化层可能减少，但全局平均池化反而越来越重要。

#### 5. 池化层仍然重要，但角色更明确
**1️⃣ 它没有消失**

虽然现代网络中有很多变化，

但这并不意味着池化层不重要了。

实际上，池化层仍然非常常见，尤其是：
* 基础 CNN
* 教学型网络
* 一些轻量模型
* 某些经典结构中

---

**2️⃣ 它的角色变得更明确**

现代网络更明确地区分了池化层的用途：
* 局部池化：用于阶段性下采样
* 全局池化：用于最后特征汇总

也就是说，不是“到处都用”，

而是“在合适的位置使用”。

#### 6. 一个简单的结构对比

##### 6.1 早期经典风格
```
Conv → ReLU → MaxPool
Conv → ReLU → MaxPool
Flatten → FC → FC
```

##### 6.2 更现代的常见风格
```
Conv(stride=2) → ReLU
Conv(stride=2) → ReLU
...
Global Average Pooling → Linear
```

#### 6.3 这说明什么
说明现代网络常见的变化是：
* 中间下采样：更可能交给 stride 卷积
* 最后汇总：更可能交给 GAP